# Machine Sensor Anomaly Detection – Synapse Analytics Notebook

This notebook performs end-to-end analysis on machine sensor telemetry stored in the
Microsoft Fabric Lakehouse.  It covers:

1. **Data Loading** – Read raw and processed sensor events from Delta tables
2. **Exploratory Data Analysis (EDA)** – Distributions, correlations, time trends
3. **Data Preprocessing** – Feature engineering, scaling, train/test split
4. **Model Training** – Isolation Forest with hyperparameter tuning
5. **Evaluation** – Metrics, confusion matrix, ROC curve
6. **Save Artifacts** – Persist model and scaler back to the Lakehouse

> **Note**: This notebook is designed to run inside a Microsoft Fabric Synapse
> Analytics workspace where a `spark` session and Lakehouse mount are pre-configured.
> It can also be run locally with minor adjustments (set `RUN_LOCAL = True`).

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RUN_LOCAL     = False   # Set True to run outside Fabric with local CSV file
LOCAL_CSV     = '../data/sample_sensor_data.csv'
LAKEHOUSE_NAME = 'SensorLakehouse'
RAW_TABLE      = 'raw_sensor_events'
PROCESSED_TABLE = 'processed_sensor_events'
MODEL_OUTPUT_PATH = 'Files/models/isolation_forest_latest'

FEATURE_COLS = ['temperature', 'vibration', 'pressure', 'humidity', 'current', 'voltage']
LABEL_COL    = 'is_simulated_anomaly'   # Ground-truth label (if present)
TEST_SIZE    = 0.2
RANDOM_STATE = 42
CONTAMINATION = 0.05   # Expected anomaly fraction for Isolation Forest

print('Configuration loaded ✅')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay, f1_score
)

import pickle, json, os
from datetime import datetime, timezone

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5)})
print('Imports complete ✅')

## 1  Data Loading

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
if RUN_LOCAL:
    df = pd.read_csv(LOCAL_CSV)
    print(f'Loaded {len(df):,} rows from local CSV')
else:
    # Inside Fabric: read Delta table via PySpark → pandas
    sdf = spark.read.format('delta').load(f'Tables/{LAKEHOUSE_NAME}/{PROCESSED_TABLE}')
    df  = sdf.toPandas()
    print(f'Loaded {len(df):,} rows from Lakehouse table "{PROCESSED_TABLE}"')

print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Schema & null check ───────────────────────────────────────────────────────
print('Shape:', df.shape)
print('\nNull counts:')
print(df.isnull().sum())
print('\nDtypes:')
print(df.dtypes)

## 2  Exploratory Data Analysis

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
available_features = [c for c in FEATURE_COLS if c in df.columns]
df[available_features].describe().round(3)

In [ ]:
# ── Distribution plots ────────────────────────────────────────────────────────
n_cols = 3
n_rows = (len(available_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

hue_col = LABEL_COL if LABEL_COL in df.columns else None

for i, col in enumerate(available_features):
    ax = axes[i]
    if hue_col:
        for label, grp in df.groupby(hue_col):
            ax.hist(grp[col].dropna(), bins=40, alpha=0.6,
                    label=f'anomaly={label}')
        ax.legend(fontsize=8)
    else:
        ax.hist(df[col].dropna(), bins=40, color='steelblue', alpha=0.75)
    ax.set_title(col.capitalize())
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sensor Value Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight')
plt.show()
print('Distribution plots saved ✅')

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = df[available_features].corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Sensor Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Time-series trend (if timestamp present) ──────────────────────────────────
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.sort_values('timestamp').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(df['timestamp'], df['temperature'], linewidth=0.5, label='Temperature')
    if LABEL_COL in df.columns:
        anomalies = df[df[LABEL_COL] == 1]
        ax.scatter(anomalies['timestamp'], anomalies['temperature'],
                   c='red', s=10, zorder=5, label='Anomaly')
    ax.set_title('Temperature over Time')
    ax.set_xlabel('Time')
    ax.set_ylabel('Temperature (°C)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('eda_time_series.png', bbox_inches='tight')
    plt.show()
else:
    print('No timestamp column – skipping time-series plot.')

In [ ]:
# ── Anomaly class balance ─────────────────────────────────────────────────────
if LABEL_COL in df.columns:
    counts = df[LABEL_COL].value_counts()
    rate   = counts.get(1, 0) / len(df)
    print(f'Total records : {len(df):,}')
    print(f'Normal        : {counts.get(0, 0):,}')
    print(f'Anomalous     : {counts.get(1, 0):,}  ({rate:.1%})')

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Normal', 'Anomaly'], [counts.get(0, 0), counts.get(1, 0)],
           color=['#2196F3', '#F44336'])
    ax.set_title('Class Balance')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('eda_class_balance.png', bbox_inches='tight')
    plt.show()
else:
    print('No label column – skipping class balance chart.')

## 3  Data Preprocessing

In [ ]:
# ── Feature selection & missing value handling ─────────────────────────────────
X = df[available_features].copy()
X = X.fillna(X.mean())   # Impute with column mean

y = df[LABEL_COL].astype(int).values if LABEL_COL in df.columns else None

print(f'Feature matrix shape : {X.shape}')
if y is not None:
    print(f'Label array shape    : {y.shape}')
    print(f'Anomaly rate         : {y.mean():.2%}')

In [ ]:
# ── Train/test split ──────────────────────────────────────────────────────────
X_arr = X.values
if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
else:
    X_train, X_test = train_test_split(
        X_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    y_train = y_test = None

print(f'Train size : {len(X_train):,}')
print(f'Test size  : {len(X_test):,}')

In [ ]:
# ── Feature scaling ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaler means  :', np.round(scaler.mean_,  3))
print('Scaler stdevs :', np.round(np.sqrt(scaler.var_), 3))
print('Scaling complete ✅')

## 4  Model Training

In [ ]:
# ── Train Isolation Forest ────────────────────────────────────────────────────
model = IsolationForest(
    n_estimators  = 200,
    max_samples   = 'auto',
    contamination = CONTAMINATION,
    max_features  = 1.0,
    random_state  = RANDOM_STATE,
    n_jobs        = -1,
)

model.fit(X_train_scaled)
print('Model training complete ✅')
print(f'Parameters : {model.get_params()}')

In [ ]:
# ── Generate predictions on the test set ─────────────────────────────────────
raw_preds   = model.predict(X_test_scaled)            # 1=normal, -1=anomaly
scores      = model.score_samples(X_test_scaled)      # lower = more anomalous
preds_binary = (raw_preds == -1).astype(int)          # 1=anomaly, 0=normal

print(f'Predicted anomalies : {preds_binary.sum():,} / {len(preds_binary):,}')
print(f'Predicted rate      : {preds_binary.mean():.2%}')

## 5  Model Evaluation

In [ ]:
# ── Classification report (requires ground-truth labels) ─────────────────────
if y_test is not None:
    print('Classification Report:')
    print(classification_report(y_test, preds_binary,
                                 target_names=['Normal', 'Anomaly']))

    f1  = f1_score(y_test, preds_binary, zero_division=0)
    auc = roc_auc_score(y_test, -scores)  # negative score = anomaly probability proxy
    print(f'F1 Score : {f1:.4f}')
    print(f'ROC AUC  : {auc:.4f}')
else:
    print('No ground-truth labels – skipping classification report.')
    print(f'Anomaly score stats: mean={scores.mean():.4f}, std={scores.std():.4f}')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
if y_test is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Confusion matrix
    cm = confusion_matrix(y_test, preds_binary)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Confusion Matrix')

    # ROC curve
    RocCurveDisplay.from_predictions(y_test, -scores, ax=axes[1],
                                      name='Isolation Forest')
    axes[1].set_title('ROC Curve')

    plt.tight_layout()
    plt.savefig('eval_confusion_roc.png', bbox_inches='tight')
    plt.show()
else:
    print('Skipping confusion matrix / ROC (no labels).')

In [ ]:
# ── Anomaly score distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
if y_test is not None:
    for label, name, colour in [(0, 'Normal', '#2196F3'), (1, 'Anomaly', '#F44336')]:
        mask = y_test == label
        ax.hist(-scores[mask], bins=50, alpha=0.6, label=name, color=colour)
    ax.legend()
else:
    ax.hist(-scores, bins=50, color='steelblue', alpha=0.75)
ax.set_title('Anomaly Score Distribution')
ax.set_xlabel('Anomaly Score (higher = more anomalous)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('eval_score_distribution.png', bbox_inches='tight')
plt.show()

## 6  Save Model Artifacts

In [ ]:
# ── Persist model to Lakehouse Files ─────────────────────────────────────────
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')

metrics = {}
if y_test is not None:
    metrics = {
        'f1_score' : round(float(f1),  4),
        'roc_auc'  : round(float(auc), 4),
        'test_size': int(len(y_test)),
    }

artefact = {
    'model'           : model,
    'scaler'          : scaler,
    'params'          : model.get_params(),
    'metrics'         : metrics,
    'trained_at'      : timestamp,
    'feature_columns' : available_features,
}

os.makedirs('models', exist_ok=True)
model_path = f'models/isolation_forest_{timestamp}.pkl'
with open(model_path, 'wb') as fh:
    pickle.dump(artefact, fh)

# Stable latest pointer
with open('models/model_latest.pkl', 'wb') as fh:
    pickle.dump(artefact, fh)

# Metadata sidecar
meta = {k: v for k, v in artefact.items() if k not in ('model', 'scaler')}
with open(f'models/metadata_{timestamp}.json', 'w') as fh:
    json.dump(meta, fh, indent=2)

print(f'Model saved to "{model_path}" ✅')
print(f'Metadata  : models/metadata_{timestamp}.json')
print(f'Metrics   : {metrics}')

In [ ]:
# ── (Fabric only) Copy model to Lakehouse Files ───────────────────────────────
if not RUN_LOCAL:
    import shutil
    dest = f'/lakehouse/default/Files/models/'
    os.makedirs(dest, exist_ok=True)
    shutil.copy(model_path, os.path.join(dest, f'isolation_forest_{timestamp}.pkl'))
    shutil.copy('models/model_latest.pkl', os.path.join(dest, 'model_latest.pkl'))
    print(f'Model copied to Lakehouse Files: {dest} ✅')
else:
    print('Running locally – model saved to ./models/ only.')

## Summary

| Step | Status |
|------|--------|
| Data Loading | ✅ |
| EDA | ✅ |
| Preprocessing | ✅ |
| Model Training | ✅ |
| Evaluation | ✅ |
| Artefact Saved | ✅ |

Next steps:
- Deploy `model_latest.pkl` to the REST API container via `deploy_to_aci.sh`.
- Refresh the Power BI dashboard to reflect updated predictions.
- Schedule this notebook to re-run weekly via the Fabric scheduler.